In [ ]:
# @title Install Dependeries
# 1. Cleanly remove any previous clone
%cd /content
!rm -rf /content/cdn

# 2. Install Node.js 22
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash >/dev/null 2>&1
!apt-get install -y nodejs >/dev/null 2>&1

print("\nNode:")
!node --version

print("\nnpm:")
!npm --version

# 3. Clone the repository and enter the directory
!git clone -q https://github.com/backblaze-b2-samples/cloudflare-b2 /content/cdn
%cd /content/cdn

# 4. Cleanly wipe out any default/broken node files inside the repo
!rm -rf node_modules package-lock.json

# 5. Install all the project's dependencies fresh
!npm install

# 6. version
!npx wrangler --version
!npm list aws4fetch


/content

Node:
v22.23.3

npm:
10.9.9
/content/cdn
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 40 packages, and audited 41 packages in 12s
⠋
⠋8 packages are looking for funding
⠋  run `npm fund` for details
⠋
found 0 vulnerabilities
⠋⠙
 ⛅️ wrangler 4.140.0
────────────────────
⠙⠙cloudflare-b2@1.2.0 /content/cdn
└── aws4fetch@1.0.20

⠙

In [ ]:
# @title 3. Overwrite index.js with optimized version
%%writefile /content/cdn/index.js
//
// Proxy Backblaze B2 S3-compatible API requests through Cloudflare Worker
//
import { AwsClient } from "aws4fetch";

const UNSIGNABLE_HEADERS = [
    "x-forwarded-proto",
    "x-real-ip",
    "accept-encoding",
    "if-match",
    "if-modified-since",
    "if-none-match",
    "if-range",
    "if-unmodified-since",
];

const HTTPS_PROTOCOL = "https:";
const HTTPS_PORT = "443";
const RANGE_RETRY_ATTEMPTS = 3;
const CDN_CACHE_TTL_SECONDS = 30 * 24 * 60 * 60;
const BROWSER_CACHE_TTL_SECONDS = 1 * 24 * 60 * 60;

function filterHeaders(headers, env) {
    const allowedHeaders = env["ALLOWED_HEADERS"]
        ? env["ALLOWED_HEADERS"].split(",").map(h => h.trim().toLowerCase())
        : null;

    return new Headers(
        Array.from(headers.entries()).filter(([key]) => {
            const lowerKey = key.toLowerCase();

            if (
                UNSIGNABLE_HEADERS.includes(lowerKey) ||
                lowerKey.startsWith("cf-")
            ) {
                return false;
            }

            if (allowedHeaders && !allowedHeaders.includes(lowerKey)) {
                return false;
            }

            return true;
        })
    );
}

function createHeadResponse(response) {
    return new Response(null, {
        headers: new Headers(response.headers),
        status: response.status,
        statusText: response.statusText,
    });
}

function addCacheAndCorsHeaders(response) {
    if (!response.ok) {
        return response;
    }

    const headers = new Headers(response.headers);

    headers.set(
        "Cloudflare-CDN-Cache-Control",
        `public, max-age=${CDN_CACHE_TTL_SECONDS}`
    );

    headers.set(
        "Cache-Control",
        `public, max-age=${BROWSER_CACHE_TTL_SECONDS}`
    );

    headers.set(
        "Access-Control-Allow-Origin",
        "*"
    );

    headers.set(
        "Access-Control-Expose-Headers",
        "Accept-Ranges, Content-Length, Content-Range, Content-Type, ETag"
    );

    headers.set(
        "Accept-Ranges",
        "bytes"
    );

    // Make sure MP4 objects have a browser-friendly content type
    // if the origin did not already provide one.
    if (
        !headers.has("Content-Type") &&
        response.url.toLowerCase().split("?")[0].endsWith(".mp4")
    ) {
        headers.set("Content-Type", "video/mp4");
    }

    return new Response(response.body, {
        status: response.status,
        statusText: response.statusText,
        headers,
    });
}

async function getCachedResponse(request, signedRequest, ctx) {
    const cache = caches.default;

    const cached = await cache.match(request);

    if (cached) {
        return cached;
    }

    const response = await fetch(signedRequest);

    if (response.ok) {
        const responseWithHeaders =
            addCacheAndCorsHeaders(response);

        ctx.waitUntil(
            cache.put(
                request,
                responseWithHeaders.clone()
            )
        );

        return responseWithHeaders;
    }

    return response;
}

function isListBucketRequest(env, path) {
    const pathSegments = path.split("/");

    return (
        (env["BUCKET_NAME"] === "$path" &&
            pathSegments.length < 2) ||
        (env["BUCKET_NAME"] !== "$path" &&
            path.length === 0)
    );
}

export default {
    async fetch(request, env, ctx) {

        // CORS preflight
        if (request.method === "OPTIONS") {
            return new Response(null, {
                status: 204,
                headers: {
                    "Access-Control-Allow-Origin": "*",
                    "Access-Control-Allow-Methods":
                        "GET, HEAD, OPTIONS",
                    "Access-Control-Allow-Headers":
                        "Range, Content-Type, If-Range, Origin",
                    "Access-Control-Expose-Headers":
                        "Accept-Ranges, Content-Length, Content-Range, Content-Type, ETag",
                    "Access-Control-Max-Age": "86400",
                },
            });
        }

        if (!["GET", "HEAD"].includes(request.method)) {
            return new Response(null, {
                status: 405,
                statusText: "Method Not Allowed",
            });
        }

        const url = new URL(request.url);

        url.protocol = HTTPS_PROTOCOL;
        url.port = HTTPS_PORT;

        let path = url.pathname
            .replace(/^\//, "")
            .replace(/\/$/, "");

        if (
            isListBucketRequest(env, path) &&
            String(env["ALLOW_LIST_BUCKET"]) !== "true"
        ) {
            return new Response(null, {
                status: 404,
                statusText: "Not Found",
            });
        }

        const rcloneDownload =
            String(env["RCLONE_DOWNLOAD"]) === "true";

        switch (env["BUCKET_NAME"]) {
            case "$path":
                url.hostname = env["B2_ENDPOINT"];
                break;

            case "$host":
                url.hostname =
                    url.hostname.split(".")[0] +
                    "." +
                    env["B2_ENDPOINT"];
                break;

            default:
                url.hostname =
                    env["BUCKET_NAME"] +
                    "." +
                    env["B2_ENDPOINT"];
                break;
        }

        const headers = filterHeaders(
            request.headers,
            env
        );

        const client = new AwsClient({
            accessKeyId:
                env["B2_APPLICATION_KEY_ID"],

            secretAccessKey:
                env["B2_APPLICATION_KEY"],

            service: "s3",
        });

        const requestMethod = request.method;

        if (rcloneDownload) {
            url.pathname =
                env["BUCKET_NAME"] === "$path"
                    ? path.replace(/^file\//, "")
                    : path.replace(/^file\/[^/]+\//, "");
        }

        // IMPORTANT:
        // The original Range header is preserved and signed by aws4fetch.
        const signedRequest = await client.sign(
            url.toString(),
            {
                method: "GET",
                headers: headers,
            }
        );

        // Range Request handling
        if (signedRequest.headers.has("range")) {

            let attempts = RANGE_RETRY_ATTEMPTS;
            let response;
            let finalResponse;

            do {
                response = await fetch(
                    signedRequest.url,
                    {
                        method: signedRequest.method,
                        headers: signedRequest.headers,
                    }
                );

                if (
                    response.headers.has(
                        "content-range"
                    )
                ) {
                    finalResponse = response;
                    break;
                }

                if (response.ok) {
                    attempts -= 1;

                    if (
                        attempts > 0 &&
                        response.body
                    ) {
                        await response.body.cancel();
                    } else {
                        finalResponse = response;
                    }

                } else {
                    finalResponse = response;
                    break;
                }

            } while (attempts > 0);

            if (!finalResponse) {
                finalResponse = response;
            }

            if (requestMethod === "HEAD") {

                if (finalResponse.body) {
                    await finalResponse.body.cancel();
                }

                return addCacheAndCorsHeaders(
                    createHeadResponse(
                        finalResponse
                    )
                );
            }

            return addCacheAndCorsHeaders(
                finalResponse
            );
        }

        // Full-File HEAD Request handling
        if (requestMethod === "HEAD") {

            const response =
                await fetch(signedRequest);

            if (response.body) {
                await response.body.cancel();
            }

            return addCacheAndCorsHeaders(
                createHeadResponse(response)
            );
        }

        // Full-File GET Request handling
        return await getCachedResponse(
            request,
            signedRequest,
            ctx
        );
    },
};


Overwriting /content/cdn/index.js


In [ ]:
# @title Deploy the clean Worker WITHOUT cache modifications
%cd /content/cdn

# Create a clean wrangler.toml using your values.
# Enter them privately in Colab; do not paste secrets into the notebook.

import os
import getpass
import subprocess

# ------------------------------------------------------------
# Credentials
# ------------------------------------------------------------

cf_token = getpass.getpass("Cloudflare API Token: ").strip()

b2_key_id = input(
    "B2 Application Key ID: "
).strip()

b2_secret = getpass.getpass(
    "B2 Application Key: "
).strip()

# ------------------------------------------------------------
# B2 configuration
# ------------------------------------------------------------

b2_endpoint = input(
    "B2 S3 Endpoint [s3.ca-east-006.backblazeb2.com]: "
).strip()

bucket_name = input(
    "B2 Bucket Name: "
).strip()

worker_name = input(
    "Worker Name: "
).strip()

# Basic validation
if not b2_key_id:
    raise ValueError("B2 Application Key ID is required.")

if not b2_secret:
    raise ValueError("B2 Application Key is required.")

if not b2_endpoint:
    raise ValueError("B2 S3 Endpoint is required.")

if not bucket_name:
    raise ValueError("B2 Bucket Name is required.")

if not worker_name:
    raise ValueError("Worker Name is required.")

# ------------------------------------------------------------
# Create clean wrangler.toml
# ------------------------------------------------------------

config = f'''name = "{worker_name}"
workers_dev = true
preview_urls = false
compatibility_date = "2026-09-20"
main = "index.js"

[vars]
B2_APPLICATION_KEY_ID = "{b2_key_id}"
B2_ENDPOINT = "{b2_endpoint}"
BUCKET_NAME = "{bucket_name}"
ALLOW_LIST_BUCKET = true
'''

with open("/content/cdn/wrangler.toml", "w", encoding="utf-8") as f:
    f.write(config)

print("\n✓ wrangler.toml created.")
print(f"  Worker: {worker_name}")
print(f"  B2 Endpoint: {b2_endpoint}")
print(f"  Bucket: {bucket_name}")
print("  Preview URLs: disabled")

# ------------------------------------------------------------
# Cloudflare authentication
# ------------------------------------------------------------

env = os.environ.copy()
env["CLOUDFLARE_API_TOKEN"] = cf_token

print("\nChecking Cloudflare authentication...")

p = subprocess.run(
    ["npx", "wrangler", "whoami"],
    cwd="/content/cdn",
    env=env,
    capture_output=True,
    text=True
)

if p.returncode != 0:

    print("Cloudflare authentication failed.")
    print((p.stderr or p.stdout)[-3000:])

else:

    print("✓ Cloudflare authentication successful.")

    # --------------------------------------------------------
    # Upload B2 secret
    # --------------------------------------------------------

    print("\nUploading B2 secret...")

    p = subprocess.run(
        ["npx", "wrangler", "secret", "put", "B2_APPLICATION_KEY"],
        cwd="/content/cdn",
        env=env,
        input=b2_secret + "\n",
        capture_output=True,
        text=True
    )

    if p.returncode != 0:

        print("Secret upload failed.")
        print((p.stderr or p.stdout)[-3000:])

    else:

        print("✓ B2 secret uploaded.")

        # ----------------------------------------------------
        # Deploy clean Worker
        # ----------------------------------------------------

        print("\nDeploying CLEAN Worker...")

        p = subprocess.run(
            ["npx", "wrangler", "deploy"],
            cwd="/content/cdn",
            env=env,
            capture_output=True,
            text=True
        )

        print((p.stdout or "")[-5000:])

        if p.stderr:
            print("\n--- Wrangler messages ---")
            print(p.stderr[-3000:])

        if p.returncode == 0:
            print("\n✓ CLEAN Worker deployment completed successfully.")
        else:
            print("\n✗ Worker deployment failed.")

/content/cdn
Cloudflare API Token: ··········
B2 Application Key ID: 003b683e064fcd50000000001
B2 Application Key: ··········
B2 S3 Endpoint [s3.ca-east-006.backblazeb2.com]: s3.eu-central-003.backblazeb2.com
B2 Bucket Name: demon03
Worker Name: test02

✓ wrangler.toml created.
  Worker: test02
  B2 Endpoint: s3.eu-central-003.backblazeb2.com
  Bucket: demon03
  Preview URLs: disabled

Checking Cloudflare authentication...
✓ Cloudflare authentication successful.

Uploading B2 secret...
✓ B2 secret uploaded.

Deploying CLEAN Worker...

 ⛅️ wrangler 4.140.0
────────────────────

Cloudflare collects anonymous telemetry about your usage of Wrangler. Learn more at https://github.com/cloudflare/workers-sdk/tree/main/packages/wrangler/telemetry.md
Total Upload: 17.89 KiB / gzip: 5.31 KiB
Worker Startup Time: 2 ms
Your Worker has access to the following bindings:
Binding                                                              Resource                  
env.B2_APPLICATION_KEY_ID ("003b683e

In [ ]:
%cd /content/cdn

import os

# Enter your Cloudflare API token privately in the notebook.
# It is NOT written into index.js.
from getpass import getpass

cf_token = getpass("Enter Cloudflare API Token: ")

os.environ["CLOUDFLARE_API_TOKEN"] = cf_token

print("✓ Cloudflare API token loaded for this Colab runtime")

/content/cdn
Enter Cloudflare API Token: ··········
✓ Cloudflare API token loaded for this Colab runtime


In [ ]:
%cd /content/cdn

!npx wrangler deploy

/content/cdn
⠙
 ⛅️ wrangler 4.138.0
────────────────────
Total Upload: 19.42 KiB / gzip: 5.56 KiB
Worker Startup Time: 2 ms
Your Worker has access to the following bindings:
Binding                      Resource                
env.B2_APPLICATION_KEY_ID    Environment Variable    
  "003b683e064fcd50000000001"
env.B2_ENDPOINT              Environment Variable    
  "s3.eu-central-003.backblazeb2.com"
env.BUCKET_NAME              Environment Variable    
  "demon03"
env.ALLOW_LIST_BUCKET        Environment Variable    
  true

Uploaded tata01 (1.85 sec)
Deployed tata01 triggers (0.56 sec)
  https://tata01.dhoom.workers.dev
Current Version ID: a9e8980f-6250-464d-baa8-97c546b83bc0
⠙

In [ ]:
import subprocess

url = "https://tata01.dhoom.workers.dev/"

p = subprocess.run(
    [
        "curl",
        "-sS",
        "-i",
        "--max-time", "30",
        url
    ],
    capture_output=True,
    text=True
)

print(p.stdout or p.stderr)

HTTP/2 200 
date: Thu, 24 Sep 2026 16:10:26 GMT
content-type: application/xml
content-length: 1398
cf-ray: a4030f27ec8a36af-YYZ
cf-cache-status: DYNAMIC
access-control-allow-origin: *
cache-control: public, max-age=86400
server: cloudflare
strict-transport-security: max-age=63072000
cloudflare-cdn-cache-control: public, max-age=2592000
x-amz-id-2: aYtE24TiYM95lXjCZNms0G2ZWYyJkGzW/
x-amz-request-id: f00cb24d7569389c
report-to: {"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=%2F1gjwr7OSJQAS%2BR8tk2jL12cxARMszXHUctVS5x92UqvwIzXu58a9n7usdzwB6E17%2BZmW7ZYN8zsr6%2FWkNZcZX3UKcKsE81Ld3nS5N8XKg3MnyHEN2BPr9Yda3zq3llogRPzv5BDPmGjMik%3D"}]}
nel: {"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}
alt-svc: h3=":443"; ma=86400

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<ListBucketResult xmlns="http://s3.amazonaws.com/doc/2006-03-01/">
    <Contents>
        <ETag>"8bfc96f9edf97e0011e57a75e010594a-28"</ETag>
        <Key>DemonS

In [ ]:
import requests
import time

url = "https://tata01.dhoom.workers.dev/oo.mp4"

tests = [
    ("Beginning", "bytes=0-1048575"),
    ("Middle", "bytes=5000000-6048575"),
    ("End", "bytes=12000000-13048575"),
]

for name, range_value in tests:
    t0 = time.perf_counter()

    r = requests.get(
        url,
        headers={"Range": range_value},
        stream=True,
        timeout=60
    )

    total = 0
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        if chunk:
            total += len(chunk)

    elapsed = time.perf_counter() - t0

    print(f"\n{name}")
    print("Range requested:", range_value)
    print("HTTP:", r.status_code)
    print("Content-Range:", r.headers.get("content-range"))
    print("Content-Length:", r.headers.get("content-length"))
    print("Bytes received:", total)
    print("Time:", round(elapsed, 3), "seconds")
    print("CF-Cache-Status:", r.headers.get("cf-cache-status"))
    print("Cache-Control:", r.headers.get("cache-control"))

    r.close()


Beginning
Range requested: bytes=0-1048575
HTTP: 206
Content-Range: bytes 0-1048575/13688438
Content-Length: 1048576
Bytes received: 1048576
Time: 0.951 seconds
CF-Cache-Status: BYPASS
Cache-Control: public, max-age=86400

Middle
Range requested: bytes=5000000-6048575
HTTP: 206
Content-Range: bytes 5000000-6048575/13688438
Content-Length: 1048576
Bytes received: 1048576
Time: 0.959 seconds
CF-Cache-Status: BYPASS
Cache-Control: public, max-age=86400

End
Range requested: bytes=12000000-13048575
HTTP: 206
Content-Range: bytes 12000000-13048575/13688438
Content-Length: 1048576
Bytes received: 1048576
Time: 0.972 seconds
CF-Cache-Status: BYPASS
Cache-Control: public, max-age=86400


In [ ]:
import requests
import time

url = "https://tata01.dhoom.workers.dev/oo.mp4"

for i in range(1, 4):
    t0 = time.perf_counter()

    r = requests.get(
        url,
        timeout=120,
        stream=True
    )

    total = 0
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        if chunk:
            total += len(chunk)

    elapsed = time.perf_counter() - t0

    print(f"\nRequest {i}")
    print("HTTP:", r.status_code)
    print("Bytes:", total)
    print("Time:", round(elapsed, 3), "seconds")
    print("CF-Cache-Status:", r.headers.get("cf-cache-status"))
    print("Age:", r.headers.get("age"))
    print("Cache-Control:", r.headers.get("cache-control"))
    print("Cloudflare-CDN-Cache-Control:",
          r.headers.get("cloudflare-cdn-cache-control"))

    r.close()


Request 1
HTTP: 200
Bytes: 13688438
Time: 0.407 seconds
CF-Cache-Status: HIT
Age: 1379
Cache-Control: public, max-age=86400
Cloudflare-CDN-Cache-Control: None

Request 2
HTTP: 200
Bytes: 13688438
Time: 0.32 seconds
CF-Cache-Status: HIT
Age: 1379
Cache-Control: public, max-age=86400
Cloudflare-CDN-Cache-Control: None

Request 3
HTTP: 200
Bytes: 13688438
Time: 0.326 seconds
CF-Cache-Status: HIT
Age: 1379
Cache-Control: public, max-age=86400
Cloudflare-CDN-Cache-Control: None
